<a href="https://colab.research.google.com/github/farankhandev/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farankhandev/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

This project investigates how machine learning can prioritize content pages whose search performance is declining.

**Research question:** Can a Random Forest model rank content pages labeled as declining more effectively than a Logistic Regression baseline?

**Decision supported:** Help human reviewers decide which pages to investigate first when review time is limited.

The model supports prioritization only. It does not prove why a page declined or guarantee that updating it will improve search performance.

### Dataset

This project uses the FlyRank internship warehouse full release hosted on Hugging Face.

### Tables used

- `dim_clients` — client-level metadata and data-history information.
- `dim_content` — content-level metadata.
- `fact_content_daily_performance` — daily search performance data.
- `fact_content_query_90d` — aggregated content/query-level signals.

### Data preparation and privacy

The analysis combines historical content performance with query-level signals to create a modeling dataset.

Client names, domains, URLs, search queries, and other identifying information are excluded from the public-facing analysis. Pseudonymous client identifiers are used only to separate clients between the training and test sets, not as model features.

### Date windows and leakage risk

The decline label compares impressions in the previous 30-day period with impressions in the latest 30-day period. Query-level signals come from the available 90-day query table. Because this window may overlap with the outcome period, potential temporal leakage is a limitation of the analysis.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

### Target definition

A content item is labeled as declining when its latest 30-day impressions are less than 80% of its previous 30-day impressions, representing a decline of more than 20%.

### Features

The model uses five features:

- `imp_prev30` — previous 30-day impressions.
- `visible_queries` — visible query count.
- `rare_share` — rare-query impression share.
- `anon_share` — anonymized-query impression share.
- `top_query_share` — top-query impression share.

Client identifiers and label-derived fields such as `trend_direction` and `trend_pct` are excluded from the model features.

### Models

The baseline is Logistic Regression. The main model is a Random Forest Classifier with 200 trees and random seed 42.

Both models use the same five features and the same train/test split for a fair comparison.

### Validation

The analysis uses `GroupShuffleSplit` to separate clients between training and testing. Approximately 75% of modeling rows are used for training and 25% for testing. The test set contains 51,702 rows from 13 clients.

### Leakage checks

Label-derived fields are excluded to reduce direct target leakage. However, the 90-day query signals may overlap with the outcome period, creating a potential temporal leakage risk. Results should therefore be interpreted cautiously.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

### Model comparison

Both models were evaluated on the same client-level holdout test set.

| Metric | Logistic Regression (Baseline) | Random Forest |
|---|---:|---:|
| ROC-AUC | 0.504 | 0.604 |
| Precision@50 | 0.460 | 0.880 |
| Precision@100 | 0.550 | 0.870 |
| Precision@500 | 0.546 | 0.840 |
| Accuracy | 65.8% | 61.3% |

The observed declining-content base rate was 63.3%.

### Interpretation

The Random Forest achieved higher ROC-AUC and Precision@K than Logistic Regression. Among the top 50 ranked items, 88% were labeled as declining, compared with 46% for the baseline.

However, Random Forest had lower overall accuracy: approximately 61.3%, compared with 65.8% for Logistic Regression. This suggests that its main value in this experiment is prioritizing items for human review rather than maximizing overall classification accuracy.

These results describe performance on the observed holdout test set. They do not guarantee future performance or prove that refreshing content will improve search results.

## 5. Limitations

*What this work cannot claim.*

This experiment has several limitations:

1. **Potential temporal leakage:** The 90-day query-level signals may overlap with the outcome period.
2. **Limited generalization evidence:** The test set contains 13 clients. Performance on other clients or future periods may differ.
3. **Lower overall accuracy:** The Random Forest achieved approximately 61.3% accuracy, compared with 65.8% for Logistic Regression.
4. **No causal evidence:** The model identifies patterns associated with the declining-content label. It does not prove why content performance declined.
5. **No guaranteed improvement:** The model has not established that refreshing a page will improve its search performance.
6. **Experimental status:** The results are based on one holdout evaluation and should not be treated as proof of production readiness.

Further work should test temporally separated data, investigate leakage, and evaluate performance on additional clients and future periods.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

### Recommended actions

1. **Review the highest-ranked pages first.** Prioritize pages with the highest Random Forest scores because the observed Precision@50 was 0.880 on the holdout test set.
2. **Investigate performance signals.** Examine historical impressions and available query-level signals for pages showing signs of decline.
3. **Review content manually.** Consider search intent, content quality, freshness, competition, and whether the page meets user needs before making changes.
4. **Use ranking when resources are limited.** The ranking can help reviewers focus on a smaller set of high-priority pages.
5. **Monitor performance after changes.** Track subsequent search performance, but do not assume that any improvement was caused by a content refresh.

### Recommended workflow

**Model ranking → Human review → Content decision → Refresh or test → Monitor performance**

### Confidence

These recommendations are based on experimental holdout results. Potential temporal leakage remains a concern, so the ranking should be treated as a directional decision-support tool rather than a proven forecasting system.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

### Required artifacts

The research paper should include the following charts and tables to make the results easier to understand and verify.

1. **Model comparison table:** Compare Logistic Regression and Random Forest using ROC-AUC, Precision@50, Precision@100, Precision@500, and accuracy.
2. **Precision@K chart:** Compare the two models' precision at K = 50, 100, and 500.
3. **Feature importance chart:** Show the Random Forest importance values for all five model features.
4. **Target distribution chart:** Show the proportions of content items labeled as declining and not declining.
5. **Evaluation summary:** Report the holdout test-set size, number of clients, validation method, and potential temporal leakage concern.

### Interpretation

All charts and tables should be generated from the analysis results rather than manually invented. Use anonymized labels and avoid publishing client names, domains, URLs, or private search queries.

The artifacts should help readers understand the model's ranking performance, important features, and limitations.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.